# Country Development Clustering for Humanitarian Aid Allocation

### Unsupervised Learning, Feature Engineering and K-Means Clustering

This project applies unsupervised learning to support the allocation of humanitarian aid across countries.

The scenario considers an international NGO that has raised approximately **$10 million** and needs to identify groups of countries with different levels of socioeconomic vulnerability.

The workflow includes:

- Exploratory Data Analysis (EDA)
- Correlation analysis
- Construction of composite health, trade and financial indicators
- Feature standardization
- K-Means clustering
- Hyperparameter selection
- Internal clustering metrics
- PCA-based visualization
- Cluster profiling
- Aid-priority interpretation


## 1. Problem and Dataset

The dataset contains socioeconomic and health-related information for **167 countries**.

The objective is not to predict a predefined label. Instead, the goal is to discover natural groups of countries with similar characteristics and use those groups to support humanitarian prioritization.

The available variables are:

| Variable | Description |
|---|---|
| `country` | Country name |
| `child_mort` | Deaths of children under 5 per 1,000 live births |
| `exports` | Exports of goods and services as a percentage of GDP per capita |
| `health` | Health expenditure as a percentage of GDP per capita |
| `imports` | Imports of goods and services as a percentage of GDP per capita |
| `income` | Net income per person |
| `inflation` | Annual growth rate of total GDP |
| `life_expec` | Life expectancy |
| `total_fer` | Fertility rate |
| `gdpp` | GDP per capita |

`country` is used only as an identifier. The remaining variables are numerical and can be used for clustering after appropriate preprocessing.


## 2. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from sklearn.decomposition import PCA

# The original project uses the Country-data.csv dataset.
df = pd.read_csv("Country-data.csv")


## 3. Exploratory Data Analysis

The first step is to inspect the dataset, summarize the numerical variables and evaluate their distributions.

Because the variables have very different units and magnitudes, direct distance-based clustering on the raw data would be inappropriate.


In [ ]:
print("Dataset shape:", df.shape)
display(df.head())

num = df.drop(columns="country")
display(num.describe().T)


### 3.1 Distribution and Outlier Analysis

The numerical variables show substantial differences in scale and distribution.

Economic variables such as `income` and `gdpp` are strongly right-skewed, with a relatively small number of countries presenting much larger values than the majority. Several variables also contain extreme observations.

These extreme values are meaningful in this context because they may represent countries with very different socioeconomic profiles rather than data-quality errors.


In [ ]:
num.plot(
    kind="density",
    subplots=True,
    layout=(3, 3),
    sharex=False,
    figsize=(12, 10),
)
plt.tight_layout()
plt.show()

num.plot(
    kind="box",
    subplots=True,
    layout=(3, 3),
    sharex=False,
    figsize=(12, 10),
)
plt.tight_layout()
plt.show()


## 4. Correlation Analysis and Feature Engineering

Nine numerical variables are available, but several of them describe closely related dimensions of development.

A correlation analysis is therefore used to identify redundancy and motivate the construction of three broader indicators.


In [ ]:
corr = num.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


The correlation matrix reveals several intuitive relationships:

- `income`, `gdpp` and `life_expec` are positively associated with development.
- `child_mort` is negatively related to `life_expec`, `income` and `gdpp`.
- `exports` and `imports` are strongly related to each other.

To reduce dimensionality while preserving interpretability, the original variables are grouped into three composite indicators:

**Health**

- `child_mort` — negative contribution
- `life_expec` — positive contribution
- `total_fer` — negative contribution
- `health` — positive contribution

**Trade**

- `exports`
- `imports`

**Finance**

- `income`
- `gdpp`
- `inflation` — negative contribution

The signs reflect whether higher values are interpreted as more or less favorable within each indicator.


In [ ]:
df_ind = pd.DataFrame()
df_ind["country"] = df["country"]

df_ind["Health"] = (
    -num["child_mort"] / num["child_mort"].mean()
    + num["life_expec"] / num["life_expec"].mean()
    - num["total_fer"] / num["total_fer"].mean()
    + num["health"] / num["health"].mean()
)

df_ind["Trade"] = (
    num["exports"] / num["exports"].mean()
    + num["imports"] / num["imports"].mean()
)

df_ind["Finance"] = (
    num["income"] / num["income"].mean()
    + num["gdpp"] / num["gdpp"].mean()
    - num["inflation"] / num["inflation"].mean()
)

display(df_ind.head())


### 4.1 Composite Indicator Distributions


In [ ]:
df_ind[["Health", "Trade", "Finance"]].plot(
    kind="density",
    subplots=True,
    figsize=(8, 6),
    sharex=False,
)
plt.tight_layout()
plt.show()


## 5. Feature Standardization

K-Means relies on Euclidean distance, so the three indicators must be placed on a comparable scale.

Standardization is applied using `StandardScaler`, producing features with approximately zero mean and unit variance.

Unlike Min-Max scaling, standardization does not force extreme observations into a fixed interval. This is useful here because extreme country profiles may carry important humanitarian information.


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df_ind[["Health", "Trade", "Finance"]]
)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=["Health", "Trade", "Finance"],
)

display(X_scaled.head())


## 6. K-Means Model Selection

K-Means is selected because the final feature space contains three continuous standardized indicators and the objective is to obtain compact, interpretable country groups.

The number of clusters is evaluated from **k = 2 to k = 8** using:

- Inertia / Elbow Method
- Silhouette Score
- Calinski-Harabasz Score
- Davies-Bouldin Score

The final choice balances internal clustering quality with interpretability for humanitarian decision-making.


In [ ]:
rows = []

for k in range(2, 9):
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50,
    )
    labels = km.fit_predict(X_scaled)

    rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })

scores = pd.DataFrame(rows)
display(scores)

plt.figure(figsize=(6, 4))
plt.plot(scores["k"], scores["inertia"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(scores["k"], scores["silhouette"], marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs. k")
plt.show()


### 6.1 Selected Number of Clusters

The original analysis selects **k = 4** as a practical compromise between cluster separation and interpretability.

Four groups also provide a useful operational structure for distinguishing different levels of humanitarian need.


In [ ]:
k_best = 4

kmeans = KMeans(
    n_clusters=k_best,
    random_state=42,
    n_init=50,
)

clusters = kmeans.fit_predict(X_scaled)

df_result = df_ind.copy()
df_result[["Health", "Trade", "Finance"]] = X_scaled
df_result["cluster"] = clusters

display(df_result.head())


## 7. Cluster Evaluation

The final four-cluster solution is evaluated using three internal clustering metrics.

In the original execution, the model obtained:

- **Silhouette Score:** 0.3761
- **Calinski-Harabasz Score:** 111.06
- **Davies-Bouldin Score:** 0.8279

These values indicate a meaningful but not perfectly separated clustering structure, which is reasonable for socioeconomic country data where group boundaries are naturally gradual rather than absolute.


In [ ]:
sil = silhouette_score(X_scaled, clusters)
ch = calinski_harabasz_score(X_scaled, clusters)
db = davies_bouldin_score(X_scaled, clusters)

print(f"Silhouette Score: {sil:.4f}")
print(f"Calinski-Harabasz Score: {ch:.2f}")
print(f"Davies-Bouldin Score: {db:.4f}")


### 7.1 PCA Visualization

PCA is used only for visualization. It does **not** influence the K-Means training or cluster assignments.


In [ ]:
pca2 = PCA(n_components=2)
X2 = pca2.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X2[:, 0], X2[:, 1], c=clusters)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Country Clusters — 2D PCA Projection")
plt.show()

pca3 = PCA(n_components=3)
X3 = pca3.fit_transform(X_scaled)

df_plot = pd.DataFrame(
    X3,
    columns=["PC1", "PC2", "PC3"],
)
df_plot["cluster"] = clusters.astype(str)
df_plot["country"] = df["country"]

fig = px.scatter_3d(
    df_plot,
    x="PC1",
    y="PC2",
    z="PC3",
    color="cluster",
    hover_name="country",
    title="Country Clusters — 3D PCA Projection",
)
fig.show()


## 8. Cluster Interpretation and Aid Prioritization

Cluster identifiers produced by K-Means are arbitrary. A cluster numbered `0`, `1`, `2` or `3` does not inherently represent a specific aid level.

For that reason, the clusters should first be profiled using the variables that the NGO considers particularly relevant: **income** and **child mortality**.

Aid priority is then assigned from the observed cluster profiles rather than from the numeric cluster ID itself.


In [ ]:
df_eval = df.copy()
df_eval["cluster"] = clusters

cluster_profile = (
    df_eval
    .groupby("cluster")
    .agg(
        countries=("country", "count"),
        mean_income=("income", "mean"),
        median_income=("income", "median"),
        mean_child_mort=("child_mort", "mean"),
        median_child_mort=("child_mort", "median"),
    )
)

display(cluster_profile)


In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df_eval, x="cluster", y="income")
plt.title("Income by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Income")
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df_eval, x="cluster", y="child_mort")
plt.title("Child Mortality by Cluster")
plt.xlabel("Cluster")
plt.ylabel("Child Mortality")
plt.show()


### 8.1 Data-Driven Aid Priority

To avoid manually attaching meaning to arbitrary K-Means labels, the following code creates a vulnerability score from cluster-level income and child mortality:

- Lower income increases vulnerability.
- Higher child mortality increases vulnerability.

The four clusters are then ranked from **highest** to **lowest** humanitarian priority.


In [ ]:
profile = cluster_profile.copy()

profile["income_vulnerability"] = (
    profile["mean_income"].max() - profile["mean_income"]
) / (
    profile["mean_income"].max() - profile["mean_income"].min()
)

profile["mortality_vulnerability"] = (
    profile["mean_child_mort"] - profile["mean_child_mort"].min()
) / (
    profile["mean_child_mort"].max() - profile["mean_child_mort"].min()
)

profile["vulnerability_score"] = (
    profile["income_vulnerability"]
    + profile["mortality_vulnerability"]
) / 2

priority_labels = [
    "Highest priority",
    "High priority",
    "Moderate priority",
    "Lowest priority",
]

priority_order = (
    profile["vulnerability_score"]
    .sort_values(ascending=False)
    .index
)

cluster_priority = {
    cluster: priority_labels[rank]
    for rank, cluster in enumerate(priority_order)
}

df_eval["aid_priority"] = df_eval["cluster"].map(cluster_priority)

display(
    profile
    .sort_values("vulnerability_score", ascending=False)
)


### 8.2 Geographic Visualization


In [ ]:
fig = px.choropleth(
    df_eval,
    locations="country",
    locationmode="country names",
    color="aid_priority",
    hover_name="country",
    title="Humanitarian Aid Priority by Country",
)

fig.show()


## 9. Conclusions

This project demonstrates how unsupervised learning can support humanitarian decision-making by transforming multiple socioeconomic variables into interpretable country groups.

The main findings are:

- Raw socioeconomic variables differ substantially in scale and distribution, making preprocessing essential.
- Correlation analysis reveals strong redundancy among several development indicators.
- Grouping the original variables into **Health, Trade and Finance** dimensions simplifies the clustering problem while preserving interpretability.
- K-Means with **four clusters** provides a useful segmentation of countries into distinct socioeconomic profiles.
- The final model achieved a **Silhouette Score of 0.3761**, a **Calinski-Harabasz Score of 111.06** and a **Davies-Bouldin Score of 0.8279**.
- Cluster labels themselves are arbitrary; humanitarian meaning should be assigned only after inspecting the underlying socioeconomic profiles.
- Income and child mortality provide a practical basis for ranking clusters according to vulnerability.

### Limitations and Potential Improvements

The analysis is based on a static dataset and therefore does not capture changes in country conditions over time.

Additional improvements could include:

- Adding education, inequality, conflict and access-to-basic-services indicators.
- Comparing K-Means with hierarchical clustering or density-based methods.
- Testing cluster stability across multiple samples or time periods.
- Comparing the resulting groups with external indicators such as the Human Development Index.
- Incorporating expert humanitarian knowledge when defining final intervention priorities.

The clustering should therefore be interpreted as a **decision-support tool**, not as an automatic allocation rule.
